# Business Task — Ask

## Scenario

As a junior data analyst working for a business intelligence consultant, I have
been tasked with leading an end-to-end analytics project for a new client.

The client is an amateur chess player who shared six years of their Chess.com
game history and asked a question they found uncomfortable to answer on their own:

---

## Business Question

> **Do the behavioral patterns in this player's Chess.com history — volume,
> timing, session structure, and requeue habits — show signs of compulsive use?**

---

## Why This Question

The client noticed they were sometimes playing chess without remembering opening
the app. Rather than relying on intuition, they wanted a data-driven answer: is
this a casual hobby, or does the data tell a different story?

---

## Key Questions

- Are there binge days where the user played far more than usual?
- Does the user requeue faster after certain results — win, draw, or loss?
- How much of the user's play happens late at night (after 22:00)?
- How long are individual playing sessions, and do the longest ones show
  performance decline?
- Is the yearly volume of games accelerating, stable, or declining over six years?

---

## Metrics

| Metric | Description |
|---|---|
| `score` | Numeric outcome: win=1, draw=0.5, loss=0 |
| `games` | Volume of games per time segment or session |
| `gap_minutes` | Time between end of one game and start of the next |
| `session_id` | Identifier for a continuous block of games (gap < 2 hours) |
| `binge_day` | Any day at or above the 95th percentile of daily game volume |

---

## Stakeholders

- **Primary:** The player — uses insights to make practical changes to their
  playing habits
- **Secondary:** Any analyst applying the same behavioral methodology to other
  players or to other mobile apps with similar engagement patterns

---

## Data Source

All data was collected from the **Chess.com Public API**
(`https://api.chess.com/pub/`), which provides full game history, player stats,
and opponent profiles at no cost and without authentication.

The dataset covers **21,007 games** played between **November 2020 and May 2026**,
across **20,242 unique opponents** from **223 countries**.

---

## Project Structure

| Notebook | Purpose |
|---|---|
| `01_data_collection` | Pull game history and opponent profiles from the Chess.com API |
| `02_data_cleaning` | Clean timestamps, standardize results, engineer features |
| `03_exploratory_analysis` | Five behavioral analyses with visualizations |
| `04_visualizations` | Publication-ready charts for portfolio and social posts |

---



# Next Steps

The next notebook will focus on:
- cleaning timestamps and extracting time features
- standardizing game results across all draw and loss subtypes
- engineering player-centric columns (side, rating, opponent)
- preparing the dataset for behavioral analysis


## Imports

In [1]:
import requests
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from datetime import datetime
import time

## User configuration

The Chess.com username is defined once in `src/config.py` and imported here.

In [2]:
PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import USERNAME

## Project paths

In [3]:
RAW_DATA_DIR = PROJECT_ROOT / "dataset" / "raw" / USERNAME

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR

WindowsPath('D:/Data Analisi/notebooks/scacchi/chesscom-personal-analysis/dataset/raw/elmurie')

# API helper

Chess.com requires a valid User-Agent in order to avoid 403/rate limiting.

In [ ]:
headers = {
    "User-Agent": "chess-analytics-project (contact: xxx@xxx.com)"
}

BASE_URL = f"https://api.chess.com/pub/player/{USERNAME}"


def get_json(url):

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Request failed: {url}")
        print(response.status_code)
        return None

    return response.json()

## Game Archives

Chess.com stores games in monthly archives.

The API first returns a list of archive URLs,
then each archive contains the games played during that month.

In [9]:
archives_data = get_json(
    f"{BASE_URL}/games/archives"
)

archives = archives_data["archives"]

len(archives)

67

## Download all games

In [10]:
all_games = []

for archive in archives:

    data = get_json(archive)

    if not data:
        continue

    for game in data.get("games", []):

        all_games.append({

            "date": (
                datetime.fromtimestamp(
                    game.get("end_time")
                ).strftime("%Y-%m-%d %H:%M:%S")
                if game.get("end_time")
                else None
            ),

            "url": game.get("url"),

            "time_class": game.get("time_class"),
            "time_control": game.get("time_control"),

            "white": game.get("white", {}).get("username"),
            "black": game.get("black", {}).get("username"),

            "white_rating": game.get("white", {}).get("rating"),
            "black_rating": game.get("black", {}).get("rating"),

            "white_result": game.get("white", {}).get("result"),
            "black_result": game.get("black", {}).get("result"),

            "eco": game.get("eco"),

            "pgn": game.get("pgn")
        })

## Create Dataframe

In [11]:
games_df = pd.DataFrame(all_games)

games_df.head()

,date,url,time_class,time_control,white,black,white_rating,black_rating,white_result,black_result,eco,pgn
0,2020-11-10 15:30:24,https://www.chess.com/game/live/5719328865,rapid,600,elmurie,TheMrNoName,213,412,resigned,win,https://www.chess.com/openings/Polish-Opening-...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
1,2020-11-10 15:46:19,https://www.chess.com/game/live/5719418873,rapid,600,smackersmashbot,elmurie,216,345,resigned,win,https://www.chess.com/openings/Kings-Pawn-Open...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
2,2020-11-10 16:09:31,https://www.chess.com/game/live/5719482469,rapid,600,amkh98,elmurie,371,256,win,checkmated,https://www.chess.com/openings/Kings-Pawn-Open...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
3,2020-11-10 17:39:54,https://www.chess.com/game/live/5720005454,rapid,600,elmurie,callumfindlay4,193,330,checkmated,win,https://www.chess.com/openings/Vienna-Game-Max...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
4,2020-11-10 17:51:42,https://www.chess.com/game/live/5720050101,rapid,600,callumfindlay4,elmurie,291,277,resigned,win,https://www.chess.com/openings/Caro-Kann-Defen...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."


## Dataset overview

The dataset looks pretty good!

In [12]:
games_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21007 entries, 0 to 21006
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          21007 non-null  object
 1   url           21007 non-null  object
 2   time_class    21007 non-null  object
 3   time_control  21007 non-null  object
 4   white         21007 non-null  object
 5   black         21007 non-null  object
 6   white_rating  21007 non-null  int64 
 7   black_rating  21007 non-null  int64 
 8   white_result  21007 non-null  object
 9   black_result  21007 non-null  object
 10  eco           21007 non-null  object
 11  pgn           21007 non-null  object
dtypes: int64(2), object(10)
memory usage: 1.9+ MB


In [13]:
games_df.describe(include="all")

,date,url,time_class,time_control,white,black,white_rating,black_rating,white_result,black_result,eco,pgn
count,21007,21007,21007,21007,21007,21007,21007.000000,21007.000000,21007,21007,21007,21007
unique,21007,21007,4,11,10350,10325,NaN,NaN,10,10,1450,21007
top,2020-11-10 15:30:24,https://www.chess.com/game/live/5719328865,blitz,180,elmurie,elmurie,NaN,NaN,win,win,https://www.chess.com/openings/Vienna-Game,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
freq,1,1,19825,15802,10488,10519,NaN,NaN,10663,9676,952,1
mean,NaN,NaN,NaN,NaN,NaN,NaN,628.526348,627.565954,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,120.286759,119.642032,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,100.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,561.000000,560.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,649.000000,649.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,711.000000,710.000000,NaN,NaN,NaN,NaN


In [14]:
games_df["time_class"].value_counts()

time_class
blitz     19825
rapid       653
bullet      455
daily        74
Name: count, dtype: int64

## Opponents list

We can extract the opponents' country, but we need to extract the profiles first

### Opponent normalisation 

In [ ]:
games_df["opponent"] = np.where(
    games_df["white"] == USERNAME,
    games_df["black"],
    games_df["white"]
)

games_df["opponent_clean"] = (
    games_df["opponent"]
    .dropna()
    .str.lower()
    .str.strip()
)

### Unique opponents

In [16]:
opponents = (
    games_df["opponent_clean"]
    .dropna()
    .unique()
)

print(
    f"Unique opponents: {len(opponents)}"
)

Unique opponents: 20242


### Save unique opponents

In [17]:
opponents_df = pd.DataFrame({
    "opponent": opponents
})

opponents_path = (
    RAW_DATA_DIR
    / "unique_opponents.csv"
)

opponents_df.to_csv(
    opponents_path,
    index=False
)

print(
    f"Saved: {opponents_path}"
)

Saved: D:\Data Analisi\notebooks\scacchi\chesscom-personal-analysis\dataset\raw\elmurie\unique_opponents.csv


### Opponent enrichment cache

In [18]:
profiles_path = (
    RAW_DATA_DIR
    / "opponent_profiles.csv"
)

### Load existing cache if present

In [19]:
if profiles_path.exists():

    existing_profiles = pd.read_csv(
        profiles_path
    )

    print(
        f"Loaded existing profiles: {len(existing_profiles)}"
    )

else:

    existing_profiles = pd.DataFrame()

    print(
        "No existing profile cache found."
    )

Loaded existing profiles: 15303


### Already downloaded users

In [20]:
if not existing_profiles.empty:

    downloaded_users = set(
        existing_profiles["opponent_clean"]
    )

else:

    downloaded_users = set()

### Safer get_json()

In [21]:
def get_json(url):

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        if response.status_code != 200:

            return None

        return response.json()

    except Exception as e:

        print(f"ERROR: {url}")

        print(e)

        return None

### Opponent profile enrichment

In [22]:
new_profiles = []

total = len(opponents_df)

for i, opponent in enumerate(
    opponents_df["opponent"]
):

    # skip existing users
    if opponent in downloaded_users:

        continue

    try:

        OPPONENT_URL = (
            f"https://api.chess.com/pub/player/{opponent}"
        )

        profile = get_json(OPPONENT_URL)

        if not profile:

            continue

        country_url = profile.get("country")

        country_code = None

        if country_url:

            country_code = (
                country_url
                .split("/")[-1]
            )

        new_profiles.append({

            "opponent_clean": opponent,

            "country_code": country_code,

            "title": profile.get("title"),

            "followers": profile.get("followers"),

            "joined": profile.get("joined"),

            "last_online": profile.get("last_online")
        })

        # progress
        if i % 100 == 0:

            print(
                f"{i}/{total}"
            )

        # incremental save
        if i % 100 == 0:

            temp_df = pd.concat([

                existing_profiles,

                pd.DataFrame(new_profiles)

            ]).drop_duplicates(
                subset="opponent_clean"
            )

            temp_df.to_csv(
                profiles_path,
                index=False
            )

            print(
                f"Incremental save at {i}"
            )

        time.sleep(0.5)

    except Exception as e:

        print(
            f"ERROR on {opponent}"
        )

        print(e)

        continue

15400/20242
Incremental save at 15400
15500/20242
Incremental save at 15500
15600/20242
Incremental save at 15600
15700/20242
Incremental save at 15700
15800/20242
Incremental save at 15800
15900/20242
Incremental save at 15900
16000/20242
Incremental save at 16000
16100/20242
Incremental save at 16100
16200/20242
Incremental save at 16200
16300/20242
Incremental save at 16300
16400/20242
Incremental save at 16400
16600/20242
Incremental save at 16600
16700/20242
Incremental save at 16700
16800/20242
Incremental save at 16800
16900/20242
Incremental save at 16900
17100/20242
Incremental save at 17100
17200/20242
Incremental save at 17200
17300/20242
Incremental save at 17300
17400/20242
Incremental save at 17400
17500/20242
Incremental save at 17500
17600/20242
Incremental save at 17600
17700/20242
Incremental save at 17700
17800/20242
Incremental save at 17800
17900/20242
Incremental save at 17900
18000/20242
Incremental save at 18000
18100/20242
Incremental save at 18100
18400/20242


### Final save

In [23]:
final_profiles = pd.concat([

    existing_profiles,

    pd.DataFrame(new_profiles)

]).drop_duplicates(
    subset="opponent_clean"
)

final_profiles.to_csv(
    profiles_path,
    index=False
)

print(
    f"Final profiles saved: {len(final_profiles)}"
)

Final profiles saved: 19606


## Save dataset

In [24]:
games_df.to_csv(
    RAW_DATA_DIR / "games.csv",
    index=False
)

print("Dataset saved.")

Dataset saved.
